# 从零实现 GraphCL：图增强、GIN 编码器与多正例 NT-Xent

本 Notebook 只使用 PyTorch 张量和 `nn.Module`，手写图批处理合同、节点/边/特征增强、GIN 消息传递、图级池化、投影头与对称 NT-Xent；不使用 PyG、DGL、现成 GNN、`MultiheadAttention` 或 Transformer。重点不是“跑出一个好看的准确率”，而是让随机增强、跨图隔离、假负例与发布边界都能被断言审计。

原始思想参考：[GraphCL, NeurIPS 2020](https://arxiv.org/abs/2010.13902)、[GIN, ICLR 2019](https://arxiv.org/abs/1810.00826)、[SimCLR, ICML 2020](https://arxiv.org/abs/2002.05709)。这里使用离线合成小图做受控实验，结果只证明实现链路在该 fixture 上有效，不代表真实分子图或社交图泛化。


In [ ]:
from __future__ import annotations

import copy, hashlib, json, math, random, warnings
from dataclasses import dataclass
from types import MappingProxyType
import numpy as np
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)
import torch
from torch import nn
import torch.nn.functional as F

SEED = 6101
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

def canonical_digest(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def state_descriptor(state: dict[str, torch.Tensor]) -> dict:
    result = {}
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        result[key] = {
            "dtype": str(value.dtype), "shape": list(value.shape),
            "sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest(),
        }
    return result

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1
assert canonical_digest({"b": 2, "a": 1}) == canonical_digest({"a": 1, "b": 2})
assert state_descriptor({"x": torch.tensor([1.0])}) != state_descriptor({"x": torch.tensor([2.0])})
assert not any(name in globals() for name in ("torch_geometric", "dgl"))


## 1. Packed graph batch 是第一道隔离边界

节点特征为 `x:[N,F]`，边为 `edge_index:[2,E]`，`batch:[N]` 指明每个节点属于哪张图。任何边都必须满足 `batch[src] == batch[dst]`；否则一次消息传递就会把测试样本或其他租户的信息混进来。这里不使用 padding，空图被拒绝，图编号必须从 0 连续出现。

对图置换只改变节点局部顺序，不应改变图级表示。`semantic()` 同时记录特征值、边、batch、dtype 与 shape，后面发布包装器会重新计算，而不是相信调用者声称的摘要。


In [ ]:
@dataclass(frozen=True)
class GraphBatch:
    x: torch.Tensor
    edge_index: torch.Tensor
    batch: torch.Tensor

    def validate(self) -> "GraphBatch":
        if self.x.ndim != 2 or self.x.numel() == 0 or not torch.isfinite(self.x).all():
            raise ValueError("x 必须是有限的非空 [N,F]")
        if self.edge_index.ndim != 2 or self.edge_index.shape[0] != 2 or self.edge_index.dtype != torch.long:
            raise ValueError("edge_index 必须是 long [2,E]")
        if self.batch.shape != (self.x.shape[0],) or self.batch.dtype != torch.long:
            raise ValueError("batch 必须是 long [N]")
        if self.batch.numel() and int(self.batch.min()) != 0:
            raise ValueError("图编号必须从 0 开始")
        ids = self.batch.unique(sorted=True)
        if not torch.equal(ids, torch.arange(len(ids), device=ids.device)):
            raise ValueError("图编号必须连续且每图至少一个节点")
        if self.edge_index.numel():
            src, dst = self.edge_index
            if int(src.min()) < 0 or int(dst.min()) < 0 or int(src.max()) >= len(self.x) or int(dst.max()) >= len(self.x):
                raise ValueError("边索引越界")
            if not torch.equal(self.batch[src], self.batch[dst]):
                raise ValueError("检测到跨图边")
        return self

    @property
    def num_graphs(self) -> int:
        return int(self.batch.max()) + 1

    def semantic(self) -> dict:
        self.validate()
        def desc(t):
            t = t.detach().cpu().contiguous()
            return {"dtype": str(t.dtype), "shape": list(t.shape),
                    "sha256": hashlib.sha256(t.numpy().tobytes()).hexdigest()}
        return {"x": desc(self.x), "edge_index": desc(self.edge_index), "batch": desc(self.batch)}

probe = GraphBatch(torch.eye(3), torch.tensor([[0,1,1,2],[1,0,2,1]]), torch.zeros(3, dtype=torch.long)).validate()
assert probe.num_graphs == 1 and probe.semantic()["x"]["shape"] == [3, 3]
try:
    GraphBatch(torch.ones(4,2), torch.tensor([[0],[2]]), torch.tensor([0,0,1,1])).validate()
    raise AssertionError("跨图边未被拒绝")
except ValueError as exc:
    assert "跨图" in str(exc)


## 2. 三类增强及其随机数合同

GraphCL 的两个 view 必须独立采样，但也必须可复现。所有随机操作都显式接收 `torch.Generator`：

- 节点丢弃：每张图至少保留一个节点，随后重映射边索引；
- 边丢弃：本 Notebook 把输入解释为无向图的双向 arc，因此 `(u,v)` 与 `(v,u)` 必须共享一次 Bernoulli 决策；
- 特征遮蔽：逐元素置零，保持 shape、batch 与边不变。

联合 recipe 写为 `node_drop → paired_edge_drop → feature_mask`。增强概率必须在 `[0,1)`，`0` 是 identity oracle 的合法配置。相同 seed 必须逐位复现，不同 seed 则应产生独立 view；随机种子只是随机流标识，不能充当正例 ID。


In [ ]:
def _rand(shape, generator):
    return torch.rand(shape, generator=generator)

def validate_reciprocal_arcs61(edge_index: torch.Tensor) -> None:
    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError("edge_index 必须为 [2,E]")
    pairs = list(zip(edge_index[0].tolist(), edge_index[1].tolist()))
    if len(pairs) != len(set(pairs)):
        raise ValueError("无向消息图不能含重复 arc")
    pair_set = set(pairs)
    if any(u == v or (v, u) not in pair_set for u, v in pairs):
        raise ValueError("每条无向边必须由互反 arc 表示且不含自环")

def augment_graph(g: GraphBatch, *, node_drop: float, edge_drop: float,
                  feature_mask: float, generator: torch.Generator) -> GraphBatch:
    g.validate(); validate_reciprocal_arcs61(g.edge_index)
    rates = (node_drop, edge_drop, feature_mask)
    if any(not isinstance(v, (int, float)) or not 0 <= v < 1 for v in rates):
        raise ValueError("增强率必须位于 [0,1)")
    keep = _rand((len(g.x),), generator) >= node_drop
    for graph_id in range(g.num_graphs):
        members = torch.where(g.batch == graph_id)[0]
        if not keep[members].any():
            keep[members[int(torch.randint(len(members), (1,), generator=generator))]] = True
    old_to_new = torch.full((len(g.x),), -1, dtype=torch.long)
    old_to_new[keep] = torch.arange(int(keep.sum()))
    if g.edge_index.numel():
        src, dst = g.edge_index
        candidate = torch.where(keep[src] & keep[dst])[0]
        edge_keep = torch.zeros(g.edge_index.shape[1], dtype=torch.bool)
        if candidate.numel():
            pair_key = torch.minimum(src[candidate], dst[candidate]) * len(g.x) + torch.maximum(src[candidate], dst[candidate])
            _, inverse = torch.unique(pair_key, sorted=True, return_inverse=True)
            pair_count = int(inverse.max()) + 1
            pair_keep = _rand((pair_count,), generator) >= edge_drop
            edge_keep[candidate] = pair_keep[inverse]
        edge_index = old_to_new[g.edge_index[:, edge_keep]]
    else:
        edge_index = g.edge_index.clone()
    x = g.x[keep].clone()
    x[_rand(x.shape, generator) < feature_mask] = 0.0
    augmented = GraphBatch(x, edge_index, g.batch[keep].clone()).validate()
    validate_reciprocal_arcs61(augmented.edge_index)
    return augmented

gen_a = torch.Generator().manual_seed(7)
identity = augment_graph(probe, node_drop=0, edge_drop=0, feature_mask=0, generator=gen_a)
assert torch.equal(identity.x, probe.x) and torch.equal(identity.edge_index, probe.edge_index)
v1 = augment_graph(probe, node_drop=.4, edge_drop=.3, feature_mask=.2, generator=torch.Generator().manual_seed(8))
v2 = augment_graph(probe, node_drop=.4, edge_drop=.3, feature_mask=.2, generator=torch.Generator().manual_seed(8))
assert v1.semantic() == v2.semantic() and v1.x.shape[0] >= 1
assert torch.equal(v1.batch[v1.edge_index[0]], v1.batch[v1.edge_index[1]])
view_seed_a = augment_graph(probe, node_drop=0, edge_drop=0, feature_mask=.5, generator=torch.Generator().manual_seed(101))
view_seed_b = augment_graph(probe, node_drop=0, edge_drop=0, feature_mask=.5, generator=torch.Generator().manual_seed(102))
assert not torch.equal(view_seed_a.x, view_seed_b.x)
try:
    validate_reciprocal_arcs61(torch.tensor([[0, 1], [1, 2]]))
    raise AssertionError("缺失反向 arc 未被拒绝")
except ValueError as exc:
    assert "互反" in str(exc)


## 3. 手写 GIN 层与图级池化

对节点 $v$，GIN 更新为

$$h_v^{(k+1)}=\mathrm{MLP}_k\left((1+\epsilon_k)h_v^{(k)}+\sum_{u\in\mathcal N(v)}h_u^{(k)}\right).$$

`index_add_` 显式完成邻居求和，复杂度为 $O(EH+NH^2)$，内存为 $O(NH+EH)$。图表示对各层节点表示做 mean pooling 后拼接，避免大图仅因节点数更大而获得更大范数。为了让置换 oracle 严格成立，层中不放依赖 batch 统计的 BatchNorm。


In [ ]:
def mean_pool(x: torch.Tensor, batch: torch.Tensor, num_graphs: int) -> torch.Tensor:
    out = x.new_zeros((num_graphs, x.shape[1]))
    out.index_add_(0, batch, x)
    count = torch.bincount(batch, minlength=num_graphs).clamp_min(1).to(x.dtype).unsqueeze(1)
    return out / count

class GINLayer(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(()))
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or edge_index.ndim != 2 or edge_index.shape[0] != 2:
            raise ValueError("GIN 输入 shape 非法")
        agg = torch.zeros_like(x)
        if edge_index.numel():
            src, dst = edge_index
            agg.index_add_(0, dst, x[src])
        return self.mlp((1.0 + self.eps) * x + agg)

class GINEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden: int, depth: int):
        super().__init__()
        self.input = nn.Linear(in_dim, hidden)
        self.layers = nn.ModuleList([GINLayer(hidden) for _ in range(depth)])
        self.out_dim = hidden * (depth + 1)

    def forward(self, graph: GraphBatch) -> torch.Tensor:
        graph.validate()
        h = F.relu(self.input(graph.x))
        pooled = [mean_pool(h, graph.batch, graph.num_graphs)]
        for layer in self.layers:
            h = F.relu(layer(h, graph.edge_index))
            pooled.append(mean_pool(h, graph.batch, graph.num_graphs))
        return torch.cat(pooled, dim=-1)

class ProjectionHead(nn.Module):
    def __init__(self, dim: int, proj_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, proj_dim))
    def forward(self, graph_embedding: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(graph_embedding), dim=-1)

class GraphCL(nn.Module):
    def __init__(self, in_dim=3, hidden=16, depth=2, proj_dim=12):
        super().__init__()
        self.encoder = GINEncoder(in_dim, hidden, depth)
        self.projector = ProjectionHead(self.encoder.out_dim, proj_dim)
    def forward(self, graph: GraphBatch, project: bool = True) -> torch.Tensor:
        h = self.encoder(graph)
        return self.projector(h) if project else h

model61 = GraphCL()
assert model61(probe).shape == (1, 12)
assert torch.allclose(model61(probe).norm(dim=1), torch.ones(1), atol=1e-6)


## 4. 节点置换与 view identity oracle

消息传递应对节点置换等变，图级池化应对置换不变。测试不能只比较 shape：下面同时重排特征、batch，并通过逆映射改写边索引，再比较数值。identity view 则验证增强率全零不会暗中重排或修改输入。


In [ ]:
def permute_graph(g: GraphBatch, perm: torch.Tensor) -> GraphBatch:
    if sorted(perm.tolist()) != list(range(len(g.x))):
        raise ValueError("perm 不是合法置换")
    inv = torch.empty_like(perm); inv[perm] = torch.arange(len(perm))
    return GraphBatch(g.x[perm], inv[g.edge_index], g.batch[perm]).validate()

square = GraphBatch(
    torch.tensor([[1.,0,0],[0,1,0],[0,0,1],[1,1,0]]),
    torch.tensor([[0,1,1,2,2,3,3,0],[1,0,2,1,3,2,0,3]]),
    torch.zeros(4, dtype=torch.long),
).validate()
perm = torch.tensor([2,0,3,1])
model61.eval()
with torch.no_grad():
    base = model61(square, project=False)
    moved = model61(permute_graph(square, perm), project=False)
assert torch.allclose(base, moved, atol=1e-6)
assert identity.semantic() == probe.semantic()


## 5. 对称、多正例 NT-Xent 与假负例边界

把两批 view 拼成 $Z=[Z^{(1)};Z^{(2)}]$。同一 `group_id` 的其他样本都是正例，自己从分母剔除：

$$\ell_i=-\log\frac{\sum_{p\in P(i)}\exp(s_{ip}/\tau)}{\sum_{a\ne i}\exp(s_{ia}/\tau)}.$$

这比“每行只有固定配对列”的实现更安全：若 batch 中同一原图出现多个增强，或数据去重失败，额外副本不应被当成假负例。仍需注意：不知道语义标签时，不同 ID 也可能语义相同；生产系统应做近重复聚类、采样审计或 debiased contrastive loss。


In [ ]:
def multi_positive_nt_xent(z1: torch.Tensor, z2: torch.Tensor,
                           group_ids: torch.Tensor, temperature: float) -> torch.Tensor:
    if z1.shape != z2.shape or z1.ndim != 2 or group_ids.shape != (len(z1),):
        raise ValueError("NT-Xent shape 合同不匹配")
    if not 0 < temperature <= 2 or not torch.isfinite(z1).all() or not torch.isfinite(z2).all():
        raise ValueError("temperature/embedding 非法")
    z = F.normalize(torch.cat([z1, z2]), dim=-1)
    ids = torch.cat([group_ids, group_ids])
    logits = z @ z.T / temperature
    eye = torch.eye(len(z), dtype=torch.bool)
    positive = ids[:, None].eq(ids[None, :]) & ~eye
    if not positive.any(dim=1).all():
        raise ValueError("每个 anchor 至少需要一个正例")
    logits = logits.masked_fill(eye, -torch.inf)
    log_num = torch.logsumexp(logits.masked_fill(~positive, -torch.inf), dim=1)
    log_den = torch.logsumexp(logits, dim=1)
    return (log_den - log_num).mean()

z_a = torch.tensor([[1.,0.],[0.,1.]])
z_b = torch.tensor([[1.,0.],[0.,1.]])
ids = torch.tensor([10,20])
loss_ab = multi_positive_nt_xent(z_a, z_b, ids, .5)
loss_ba = multi_positive_nt_xent(z_b, z_a, ids, .5)
expected = math.log1p(2 * math.exp(-2.0))
assert torch.allclose(loss_ab, torch.tensor(expected), atol=1e-6)
assert torch.allclose(loss_ab, loss_ba, atol=1e-7)
assert multi_positive_nt_xent(z_a, z_b, ids, .25) < loss_ab  # 更尖锐且正例占优

dup_z = torch.tensor([[1.,0.],[1.,0.],[0.,1.]])
multi = multi_positive_nt_xent(dup_z, dup_z, torch.tensor([1,1,2]), .5)
false_negative = multi_positive_nt_xent(dup_z, dup_z, torch.tensor([1,9,2]), .5)
assert multi < false_negative


## 6. 无标签直通、无跨 split 重复的受控图数据

合成任务仍用两类拓扑做 smoke test：类别 0 是环，类别 1 是星形。但节点特征由 `variant` 的独立随机流生成，与 `label` 无关；同一 variant 换标签时特征逐位相同，只有边结构改变。因此线性头只能利用 encoder 从拓扑与通用特征中学到的表示，不能读取 one-hot 标签通道。

每个原图用完整 tensor 语义摘要生成 `group_id`，两个增强 view 共享该 ID；不同原图即使标签相同也不是正例。train/test 使用不相交 variant 和独立随机流，并显式断言所有原图摘要跨 split 去重。线性评估冻结 encoder，只训练新线性头。

这是小型合成任务；真实评估仍应按 scaffold、主体、时间或来源分组，并在生成增强前完成 split，避免同一语义对象以不同序列化形式跨 split 出现。


In [ ]:
def make_features61(variant: int, n: int) -> torch.Tensor:
    generator = torch.Generator().manual_seed(SEED + 1000 + int(variant))
    position = torch.linspace(-1.0, 1.0, n)
    return torch.stack([
        torch.ones(n),
        .12 * position + .025 * torch.randn(n, generator=generator),
        .05 * torch.randn(n, generator=generator),
    ], dim=1)

def make_graph(label: int, variant: int) -> GraphBatch:
    if label not in (0, 1) or not isinstance(variant, int) or variant < 0:
        raise ValueError("label/variant 合同非法")
    n = 6 + variant % 3
    x = make_features61(variant, n)  # 特征 recipe 不读取 label
    pairs = [(i, (i + 1) % n) for i in range(n)] if label == 0 else [(0, i) for i in range(1, n)]
    arcs = [(u, v) for u, v in pairs for (u, v) in ((u, v), (v, u))]
    edge = torch.tensor(arcs, dtype=torch.long).T
    validate_reciprocal_arcs61(edge)
    return GraphBatch(x, edge, torch.zeros(n, dtype=torch.long)).validate()

def pack_graphs(graphs: list[GraphBatch]) -> GraphBatch:
    if not graphs: raise ValueError("不能打包空列表")
    xs, edges, batches, offset = [], [], [], 0
    for gid, g in enumerate(graphs):
        g.validate(); validate_reciprocal_arcs61(g.edge_index)
        xs.append(g.x); edges.append(g.edge_index + offset)
        batches.append(torch.full((len(g.x),), gid, dtype=torch.long)); offset += len(g.x)
    packed = GraphBatch(torch.cat(xs), torch.cat(edges, dim=1), torch.cat(batches)).validate()
    validate_reciprocal_arcs61(packed.edge_index)
    return packed

def graph_semantic_digest61(g: GraphBatch) -> str:
    return canonical_digest(g.semantic())

def graph_group_id61(g: GraphBatch) -> int:
    return int(graph_semantic_digest61(g)[:15], 16)

assert torch.equal(make_graph(0, 7).x, make_graph(1, 7).x)
assert not torch.equal(make_graph(0, 7).edge_index, make_graph(1, 7).edge_index)

train_variants61 = list(range(10, 22))
test_variants61 = list(range(101, 107))
train_labels61 = [i % 2 for i in range(len(train_variants61))]
test_labels61 = [i % 2 for i in range(len(test_variants61))]
train_graphs = [make_graph(label, variant) for label, variant in zip(train_labels61, train_variants61)]
test_graphs = [make_graph(label, variant) for label, variant in zip(test_labels61, test_variants61)]
train_digests61 = {graph_semantic_digest61(g) for g in train_graphs}
test_digests61 = {graph_semantic_digest61(g) for g in test_graphs}
assert len(train_digests61) == len(train_graphs) and len(test_digests61) == len(test_graphs)
assert train_digests61.isdisjoint(test_digests61)

train_group_ids61 = torch.tensor([graph_group_id61(g) for g in train_graphs], dtype=torch.long)
assert train_group_ids61.unique().numel() == len(train_graphs)
train_batch = pack_graphs(train_graphs)
model61 = GraphCL()
opt = torch.optim.Adam(model61.parameters(), lr=0.015)
history = []
for step in range(24):
    g1 = augment_graph(train_batch, node_drop=.05, edge_drop=.05, feature_mask=.04,
                       generator=torch.Generator().manual_seed(SEED + 2*step))
    g2 = augment_graph(train_batch, node_drop=.05, edge_drop=.05, feature_mask=.04,
                       generator=torch.Generator().manual_seed(SEED + 2*step + 1))
    loss = multi_positive_nt_xent(model61(g1), model61(g2), train_group_ids61, .25)
    opt.zero_grad(); loss.backward(); opt.step(); history.append(float(loss))

for p in model61.encoder.parameters(): p.requires_grad_(False)
head61 = nn.Linear(model61.encoder.out_dim, 2)
head_opt = torch.optim.Adam(head61.parameters(), lr=.05)
y_train = torch.tensor(train_labels61)
with torch.no_grad(): h_train = model61(train_batch, project=False)
for _ in range(45):
    supervised = F.cross_entropy(head61(h_train), y_train)
    head_opt.zero_grad(); supervised.backward(); head_opt.step()
with torch.no_grad():
    pred = head61(model61(pack_graphs(test_graphs), project=False)).argmax(1)
accuracy61 = float((pred == torch.tensor(test_labels61)).float().mean())
assert math.isfinite(history[-1]) and history[-1] < history[0]
assert accuracy61 >= .99
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in head61.parameters())
print({"contrastive_first": round(history[0],4), "contrastive_last": round(history[-1],4),
       "controlled_linear_eval_accuracy": accuracy61})


## 7. 训练、推理与复杂度边界

训练路径是 `原图 → 两个随机增强 → 共享 encoder/projector → NT-Xent`；下游推理通常丢弃 projector，只保留 encoder 表示。若每批共 $N$ 个节点、$E$ 条边、$B$ 张图，GIN 约为 $O(EH+NH^2)$；全批 NT-Xent 相似度矩阵为 $O(B^2D)$ 时间和 $O(B^2)$ 内存，大批量时需分块、cross-batch memory 或分布式 all-gather，并处理跨卡重复 ID。

常见失败包括：增强删空整张图、双 view 共用同一 RNG 状态、跨图边、把同源副本当负例、评估时仍使用 projector、按随机节点而不是按图/主体切分，以及用受控数据准确率宣称真实泛化。


## 8. 发布制品：带外 registry 而不是“自己给自己签名”

包内哈希只能发现传输损坏，攻击者替换权重后可以重算同一个哈希。下面的 `_TRUSTED_RELEASES61` 是发布系统在制品之外保存的信任锚。manifest 同时绑定模型 schema、训练/测试图语义、split、增强 recipe 和 state 的 key/dtype/shape/bytes。

`PublishedGraphCL` 也不信任调用方传入的 `graph_digest`：它在 `forward` 中从实际张量重算输入语义，只接受 manifest 声明的图。教学代码用内存 registry 模拟签名服务；生产应使用只读制品仓、KMS 签名、版本吊销和审计日志。


In [ ]:
CONFIG61 = {"in_dim":3, "hidden":16, "depth":2, "proj_dim":12}
RECIPE61 = {"node_drop":.05,"edge_drop":.05,"edge_drop_unit":"reciprocal-undirected-pair","feature_mask":.04,"rng":"independent torch.Generator/seed+2*step"}
SPLIT61 = {"train_variants":train_variants61,"test_variants":test_variants61,"unit":"whole_original_graph","cross_split_digest_overlap":0}

def graph_digest(g: GraphBatch) -> str:
    return canonical_digest(g.semantic())

state61 = {k:v.detach().cpu().clone() for k,v in model61.state_dict().items()}
manifest61 = {
    "schema":"packed-x[N,3]-reciprocal-edge[2,E]-batch[N]/cross_graph_forbidden/v2",
    "config":CONFIG61, "augment":RECIPE61, "split":SPLIT61, "positive_groups":train_group_ids61.tolist(),
    "train_graph":graph_digest(train_batch), "test_graph":graph_digest(pack_graphs(test_graphs)),
    "state":state_descriptor(state61),
}
artifact61 = {"release_id":"graphcl-61-v1", "manifest":manifest61, "state":state61}

def artifact_digest61(artifact: dict) -> str:
    return canonical_digest({"release_id":artifact["release_id"], "manifest":artifact["manifest"]})

_TRUSTED_RELEASES61 = MappingProxyType({"graphcl-61-v1": artifact_digest61(artifact61)})

class PublishedGraphCL(nn.Module):
    def __init__(self, model: GraphCL, allowed_graphs: tuple[str,...]):
        super().__init__(); self.model = model.eval(); self.allowed_graphs = allowed_graphs
    def forward(self, graph: GraphBatch, project: bool = False) -> torch.Tensor:
        actual = graph_digest(graph)  # 从真实输入张量重算，而非接收调用者摘要
        if actual not in self.allowed_graphs:
            raise ValueError("输入图语义不属于该 release")
        with torch.no_grad(): return self.model(graph, project=project)

def load_published61(artifact: dict) -> PublishedGraphCL:
    release_id = artifact.get("release_id")
    if release_id not in _TRUSTED_RELEASES61 or artifact_digest61(artifact) != _TRUSTED_RELEASES61[release_id]:
        raise ValueError("release 未受带外 registry 信任")
    if artifact["manifest"]["state"] != state_descriptor(artifact["state"]):
        raise ValueError("state 的 key/dtype/shape/bytes 不匹配")
    if artifact["manifest"]["schema"] != "packed-x[N,3]-reciprocal-edge[2,E]-batch[N]/cross_graph_forbidden/v2":
        raise ValueError("schema 不匹配")
    cfg = artifact["manifest"]["config"]
    restored = GraphCL(**cfg); restored.load_state_dict(artifact["state"], strict=True)
    return PublishedGraphCL(restored, (artifact["manifest"]["train_graph"], artifact["manifest"]["test_graph"]))

published61 = load_published61(artifact61)
assert published61(train_batch).shape == (12, model61.encoder.out_dim)
tampered61 = copy.deepcopy(artifact61)
tampered61["state"][next(iter(tampered61["state"]))].view(-1)[0] += 1
tampered61["manifest"]["state"] = state_descriptor(tampered61["state"])
try:
    load_published61(tampered61)
    raise AssertionError("整体重签攻击未被拒绝")
except ValueError as exc:
    assert "registry" in str(exc)
try:
    published61(make_graph(0, 99))
    raise AssertionError("未发布输入语义未被拒绝")
except ValueError as exc:
    assert "输入图语义" in str(exc)


## 9. 工程检查清单与生产差距

上线前至少补齐：真实领域增强的因果合理性；增强强度消融；图/主体/时间级隔离；跨卡正例 ID 对齐；大 batch 相似度分块；近重复假负例审计；多 seed 置信区间；OOM/空图/超大图降级；权重签名与回滚；输入 schema、特征词典和预处理版本监控。

本例没有声称“GraphCL 在任意数据上有效”。它验证的是：手写算子遵循置换与隔离合同，多正例目标数值正确，随机增强可复现，受控训练能学习，并且发布加载不把调用方自签摘要当成信任来源。


In [ ]:
assert len(_TRUSTED_RELEASES61) == 1
assert set(manifest61) == {"schema","config","augment","split","positive_groups","train_graph","test_graph","state"}
assert accuracy61 == 1.0
print("GraphCL 61：所有数学、隔离、训练与发布 oracle 通过。")
